In [1]:
import pandas as pd

import asyncio
import httpx
from datetime import datetime, time, timedelta
import numpy as np

import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
from sqlalchemy.dialects.postgresql import insert

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [2]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )
    
async def extraer_datos(endpoint, fecha_ini, fecha_fin):
    url = f"http://10.1.0.103:9080/Utea/{endpoint}"
    params = {
        "pStrFecIni": fecha_ini,
        "pStrFecFin": fecha_fin,
    }
    timeout = httpx.Timeout(10.0)
    now = datetime.now()
    formatted_now = now.strftime('%d/%m/%Y %H:%M:%S')
    try:
        async with httpx.AsyncClient(timeout=timeout) as client:
            response = await client.get(url, params=params)
            if response.status_code == 200:
                print("✅ " + formatted_now + f" {endpoint}: GET exitoso")
                return response.json()
            else:
                print("⚠️ " + formatted_now + f" {endpoint}: Error en obtener respuesta")
                return {}
    except httpx.RequestError as e:
        print("❌ " + formatted_now + f" {endpoint}: Error en conexion.")
        return {}


In [3]:
async def get_datos_balanza(fecha_inicio=None, fecha_fin=None):
    # Si no se pasan fechas, se calculan por defecto como las tenías antes
    if fecha_inicio is None:
        fecha_inicio = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')
    if fecha_fin is None:
        fecha_fin = (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')
        
    # Extraer datos usando los parámetros de fecha_inicio (antes ayer) y fecha_fin (antes hoy)
    TrafCamBalanza = await extraer_datos("TrafCamBalanza", fecha_inicio, fecha_fin)
    
    df_trafcambalanza = pd.DataFrame(TrafCamBalanza)
    return df_trafcambalanza

def upsert_method(table, conn, keys, data_iter):
    """
    Función personalizada para permitir UPSERT en el to_sql de Pandas
    """
    # Convertir los datos a insertar en una lista de diccionarios
    data = [dict(zip(keys, row)) for row in data_iter]
    
    # Crear la sentencia INSERT de SQLAlchemy apuntando a tu tabla
    insert_stmt = insert(table.table).values(data)
    
    # Definir la lógica de actualización en caso de conflicto con la llave primaria
    upsert_stmt = insert_stmt.on_conflict_do_update(
        index_elements=['ticketId'], # La columna que es Llave Primaria
        set_={c.name: c for c in insert_stmt.excluded if c.name != 'ticketId'} # Actualiza el resto
    )
    
    # Ejecutar en la base de datos
    conn.execute(upsert_stmt)

In [4]:
PATH_OUTPUT_DATA = 'G:\Ingenio Azucarero Guabira S.A\COOR_GERENCIA_CANA - Parte_Horarios'

In [43]:
fecha_inicio="2026-06-08"
fecha_fin="2026-06-09"
balanza = await get_datos_balanza(fecha_inicio, fecha_fin)

✅ 10/06/2026 10:16:29 TrafCamBalanza: GET exitoso


In [44]:
# 1. Convertimos a numérico, forzando los errores a convertirse en NaN
valores_numericos = pd.to_numeric(balanza['ticketId'], errors='coerce')
# 2. Filtramos el DataFrame original: nos quedamos solo con los que NO son NaN
#    y aseguramos que sean enteros exactos (evitando decimales si los hubiera)
df_limpio = balanza[valores_numericos.notna() & (valores_numericos % 1 == 0)].copy()
# 3. Ahora que está limpio, convertimos la columna a tipo entero con seguridad
df_limpio['ticketId'] = df_limpio['ticketId'].astype(int)

In [45]:
engine = obtener_engine()
# Así es como aplicas tu función
df_limpio.to_sql(
    name='trafcambalanza', 
    con=engine, 
    if_exists='append', # 'append' es obligatorio para que use el método personalizado
    schema='datos_iag',
    index=False, 
    method=upsert_method # <-- Aquí llamas a tu función
)